# ScanNet Pretrain Dataset — Verification Notebook

End-to-end verification of the preprocessed ScanNet pretrain dataset.
Uses multiprocessing for fast parallel validation of ~100K+ tensor files.

**Checks performed:**
1. Setup & configuration
2. Mount Drive & extract dataset from tar.gz
3. Directory structure & metadata files
4. File discovery & RGB/depth pairing
5. Class-folder consistency (no rogue/missing folders)
6. Frame uniqueness (no duplicates across class folders)
7. Data leakage (scene-level overlap between splits)
8. Tensor integrity (shapes, dtypes, value ranges) — parallel
9. Class distribution & balance
10. Normalization stats recomputation & verification — parallel
11. Depth statistics (zero fraction, range sanity) — parallel
12. Dataset loader smoke test
13. Visual spot check
14. Summary

## 1. Setup

In [ ]:
import glob
import json
import multiprocessing
import os
import re
import shutil
import subprocess
import sys
import time
from collections import Counter, defaultdict
from functools import partial

import matplotlib.pyplot as plt
import numpy as np
import torch

# ── Configuration ──────────────────────────────────────────────────────
DATA_ROOT = '/content/scannet_pretrain_256'
DRIVE     = '/content/drive/MyDrive/datasets'
DRIVE_TAR = os.path.join(DRIVE, 'scannet_pretrain_256.tar.gz')

EXPECTED_SPATIAL = 256          # Stored tensor spatial size
EXPECTED_NUM_CLASSES = 20
MAX_WORKERS = min(os.cpu_count() or 1, 8)
# Tolerance for norm-stats recomputation (absolute)
STATS_ATOL = 0.005

print(f'Data root:   {DATA_ROOT}')
print(f'Drive tar:   {DRIVE_TAR}')
print(f'Workers:     {MAX_WORKERS}')
print(f'Stats tol:   {STATS_ATOL}')


## 2. Mount Drive & Extract Dataset

The preprocessed dataset is stored as `scannet_pretrain_256.tar.gz` on
Google Drive. This cell mounts Drive and extracts to local disk for
fast I/O during verification.

In [ ]:
from google.colab import drive

# Mount Drive (skip if already mounted)
if not os.path.isdir('/content/drive/MyDrive'):
    drive.mount('/content/drive')
else:
    print('Drive already mounted')

# Extract tar.gz to local disk (skip if already extracted)
if os.path.exists(os.path.join(DATA_ROOT, 'train')):
    train_n = sum(1 for _ in glob.glob(
        os.path.join(DATA_ROOT, 'train', '*', '*_rgb.pt')))
    val_n = sum(1 for _ in glob.glob(
        os.path.join(DATA_ROOT, 'val', '*', '*_rgb.pt')))
    print(f'Dataset already on local disk: '
          f'{train_n:,} train + {val_n:,} val = {train_n + val_n:,} frames')
elif os.path.exists(DRIVE_TAR):
    tar_size = os.path.getsize(DRIVE_TAR) / (1024 ** 3)
    print(f'Extracting {DRIVE_TAR} ({tar_size:.1f} GB) to local disk...')
    t0 = time.time()
    subprocess.run([
        'tar', 'xzf', DRIVE_TAR,
        '-C', os.path.dirname(DATA_ROOT),
    ], check=True)
    elapsed = time.time() - t0
    train_n = sum(1 for _ in glob.glob(
        os.path.join(DATA_ROOT, 'train', '*', '*_rgb.pt')))
    val_n = sum(1 for _ in glob.glob(
        os.path.join(DATA_ROOT, 'val', '*', '*_rgb.pt')))
    print(f'Extracted in {elapsed:.0f}s: '
          f'{train_n:,} train + {val_n:,} val = {train_n + val_n:,} frames')
else:
    raise FileNotFoundError(
        f'Dataset not found at {DATA_ROOT} or {DRIVE_TAR}.\n'
        f'Run the preprocessing notebook first.'
    )


## 3. Directory Structure & Metadata

In [ ]:
# ── 2a. Check top-level structure ─────────────────────────────────────
errors = []

for required in ['train', 'val', 'class_names.txt', 'norm_stats.json']:
    path = os.path.join(DATA_ROOT, required)
    exists = os.path.exists(path)
    status = 'OK' if exists else 'MISSING'
    print(f'  [{status}] {required}')
    if not exists:
        errors.append(f'Missing: {required}')

assert not errors, f'Structure errors: {errors}'
print('\nDirectory structure: PASS')

# ── 2b. Load and validate class_names.txt ─────────────────────────────
with open(os.path.join(DATA_ROOT, 'class_names.txt')) as f:
    class_names = [line.strip() for line in f if line.strip()]
    # Handle indexed format "0: bathroom"
    class_names = [
        name.split(': ', 1)[1] if ': ' in name and name.split(': ', 1)[0].isdigit()
        else name
        for name in class_names
    ]

print(f'\nClasses ({len(class_names)}):')
for i, name in enumerate(class_names):
    print(f'  {i:2d}: {name}')

assert len(class_names) == EXPECTED_NUM_CLASSES, (
    f'Expected {EXPECTED_NUM_CLASSES} classes, got {len(class_names)}'
)
assert len(class_names) == len(set(class_names)), 'Duplicate class names!'
print(f'\nClass names: PASS ({len(class_names)} unique classes)')

# ── 2c. Load and validate norm_stats.json ─────────────────────────────
with open(os.path.join(DATA_ROOT, 'norm_stats.json')) as f:
    norm_stats = json.load(f)

required_keys = {'rgb_mean', 'rgb_std', 'depth_mean', 'depth_std'}
assert set(norm_stats.keys()) >= required_keys, (
    f'norm_stats.json missing keys: {required_keys - set(norm_stats.keys())}'
)
assert len(norm_stats['rgb_mean']) == 3, f'rgb_mean should have 3 elements'
assert len(norm_stats['rgb_std']) == 3, f'rgb_std should have 3 elements'
assert len(norm_stats['depth_mean']) == 1, f'depth_mean should have 1 element'
assert len(norm_stats['depth_std']) == 1, f'depth_std should have 1 element'

# Sanity: RGB stats should be in [0, 1] range
for key in ['rgb_mean', 'rgb_std']:
    for v in norm_stats[key]:
        assert 0.0 < v < 1.0, f'{key} value {v} outside (0, 1) — wrong scale?'

# Sanity: depth stats should be in meters (reasonable indoor range)
assert 0.5 < norm_stats['depth_mean'][0] < 6.0, (
    f'depth_mean={norm_stats["depth_mean"][0]} — expected 0.5-6.0m for indoor'
)
assert 0.1 < norm_stats['depth_std'][0] < 5.0, (
    f'depth_std={norm_stats["depth_std"][0]} — expected 0.1-5.0m for indoor'
)

print(f'\nnorm_stats.json:')
for k, v in norm_stats.items():
    print(f'  {k}: {v}')
print('\nNorm stats: PASS')

## 4. Discover All Files

In [ ]:
# Build a complete inventory of all .pt files, organized by split/class/scene
file_inventory = {}  # split -> class -> list of (rgb_path, depth_path, scene_id)
scene_to_split = {}  # scene_id -> set of splits it appears in
all_rgb_paths = {'train': [], 'val': []}

for split in ['train', 'val']:
    split_dir = os.path.join(DATA_ROOT, split)
    file_inventory[split] = {}

    for cls in sorted(os.listdir(split_dir)):
        cls_dir = os.path.join(split_dir, cls)
        if not os.path.isdir(cls_dir):
            continue

        rgb_stems = {}
        depth_stems = {}
        for fname in os.listdir(cls_dir):
            if fname.endswith('_rgb.pt'):
                stem = fname[:-len('_rgb.pt')]
                rgb_stems[stem] = os.path.join(cls_dir, fname)
            elif fname.endswith('_depth.pt'):
                stem = fname[:-len('_depth.pt')]
                depth_stems[stem] = os.path.join(cls_dir, fname)

        # Check pairing
        rgb_only = set(rgb_stems) - set(depth_stems)
        depth_only = set(depth_stems) - set(rgb_stems)
        assert not rgb_only, f'Unpaired RGB in {cls_dir}: {sorted(rgb_only)[:5]}'
        assert not depth_only, f'Unpaired depth in {cls_dir}: {sorted(depth_only)[:5]}'

        pairs = []
        for stem in sorted(rgb_stems):
            rgb_path = rgb_stems[stem]
            depth_path = depth_stems[stem]
            # Extract scene_id from filename
            match = re.match(r'(scene\d+_\d+)_f\d+', stem)
            scene_id = match.group(1) if match else stem
            pairs.append((rgb_path, depth_path, scene_id))
            all_rgb_paths[split].append(rgb_path)

            scene_to_split.setdefault(scene_id, set()).add(split)

        file_inventory[split][cls] = pairs

train_count = len(all_rgb_paths['train'])
val_count = len(all_rgb_paths['val'])
total_count = train_count + val_count

print(f'Train:  {train_count:>7,} frame pairs')
print(f'Val:    {val_count:>7,} frame pairs')
print(f'Total:  {total_count:>7,} frame pairs')
print(f'Scenes: {len(scene_to_split):>7,}')
print(f'\nAll files paired: PASS')

## 5. Class-Folder Consistency

Verifies that every subfolder in `train/` and `val/` matches an entry in
`class_names.txt`. Rogue folders (typos, leftovers) would be silently
ignored by the dataloader, causing silent sample loss.

In [ ]:
class_name_set = set(class_names)

for split in ['train', 'val']:
    split_dir = os.path.join(DATA_ROOT, split)
    folders_on_disk = {
        d for d in os.listdir(split_dir)
        if os.path.isdir(os.path.join(split_dir, d))
    }

    # Folders not in class_names.txt (would be silently ignored by loader)
    rogue = folders_on_disk - class_name_set
    if rogue:
        # Count how many samples would be lost
        lost = 0
        for folder in rogue:
            lost += len([
                f for f in os.listdir(os.path.join(split_dir, folder))
                if f.endswith('_rgb.pt')
            ])
        print(f'FAIL: {split}/ has {len(rogue)} rogue folders '
              f'not in class_names.txt ({lost} samples would be dropped):')
        for r in sorted(rogue):
            print(f'  {r}')
        raise ValueError(f'Rogue folders in {split}/: {sorted(rogue)}')

    # Classes in class_names.txt with no folder on disk
    missing = class_name_set - folders_on_disk
    if missing:
        print(f'WARNING: {split}/ missing folders for: {sorted(missing)}')
    else:
        print(f'{split}: all {len(class_name_set)} class folders present, '
              f'no rogue folders')

print(f'\nClass-folder consistency: PASS')


## 6. Frame Uniqueness

Verify no frame appears in multiple class folders within a split.
This would indicate a preprocessing bug where the same scene was
classified into two different categories.

In [ ]:
for split in ['train', 'val']:
    stem_to_class = {}  # stem -> class folder it was found in
    duplicates = []     # (stem, class1, class2)

    for cls, pairs in file_inventory[split].items():
        for rgb_path, _, _ in pairs:
            stem = os.path.basename(rgb_path)[:-len('_rgb.pt')]
            if stem in stem_to_class:
                duplicates.append((stem, stem_to_class[stem], cls))
            else:
                stem_to_class[stem] = cls

    if duplicates:
        print(f'FAIL: {split}/ has {len(duplicates)} duplicate frames '
              f'across class folders:')
        for stem, cls1, cls2 in duplicates[:10]:
            print(f'  {stem} -> {cls1} AND {cls2}')
        raise ValueError(f'{len(duplicates)} duplicate frames in {split}/')
    else:
        print(f'{split}: {len(stem_to_class):,} unique frames, no duplicates')

print(f'\nFrame uniqueness: PASS')


## 7. Data Leakage Test

In [ ]:
# No scene should appear in both train and val
leaked = [sid for sid, splits in scene_to_split.items() if len(splits) > 1]

if leaked:
    print(f'DATA LEAKAGE: {len(leaked)} scenes in both splits!')
    for s in sorted(leaked)[:20]:
        print(f'  {s}')
    raise ValueError(f'Data leakage: {len(leaked)} scenes')

train_scenes = sum(1 for s in scene_to_split.values() if 'train' in s)
val_scenes = sum(1 for s in scene_to_split.values() if 'val' in s)
print(f'Train scenes: {train_scenes}')
print(f'Val scenes:   {val_scenes}')
print(f'Overlap:      0')
print(f'\nData leakage: PASS')

## 8. Parallel Tensor Integrity Check

Validates every `.pt` file for correct shape, dtype, and value ranges.
Uses multiprocessing to scan all files in parallel.

In [ ]:
def validate_pair(rgb_path, expected_size=256):
    """Validate a single RGB/depth pair. Returns (status, error_msg_or_None)."""
    depth_path = rgb_path.replace('_rgb.pt', '_depth.pt')
    try:
        rgb = torch.load(rgb_path, weights_only=True)
        depth = torch.load(depth_path, weights_only=True)

        # Shape
        if rgb.shape != (3, expected_size, expected_size):
            return ('shape', f'{rgb_path}: RGB shape {rgb.shape}')
        if depth.shape != (1, expected_size, expected_size):
            return ('shape', f'{depth_path}: Depth shape {depth.shape}')

        # Dtype
        if rgb.dtype != torch.uint8:
            return ('dtype', f'{rgb_path}: RGB dtype {rgb.dtype}')
        if depth.dtype not in (torch.uint16, torch.int16):
            return ('dtype', f'{depth_path}: Depth dtype {depth.dtype}')

        # Value ranges
        if rgb.max() == 0:
            return ('value', f'{rgb_path}: RGB all zeros')
        # Depth: allow all-zero frames (sensor dropout), but flag them
        # Check for unreasonable max depth (>65m = 65000mm)
        depth_max = depth.to(torch.int32).max().item()
        if depth_max > 65000:
            return ('value', f'{depth_path}: depth max {depth_max}mm (>65m)')

        return ('ok', None)
    except Exception as e:
        return ('corrupt', f'{rgb_path}: {e}')


# Collect all RGB paths for validation
all_paths = all_rgb_paths['train'] + all_rgb_paths['val']
print(f'Validating {len(all_paths):,} frame pairs with {MAX_WORKERS} workers...')

validate_fn = partial(validate_pair, expected_size=EXPECTED_SPATIAL)

t0 = time.time()
with multiprocessing.Pool(MAX_WORKERS) as pool:
    results = pool.map(validate_fn, all_paths, chunksize=256)
elapsed = time.time() - t0

# Tally results
status_counts = Counter(r[0] for r in results)
failures = [(s, msg) for s, msg in results if s != 'ok']

print(f'\nCompleted in {elapsed:.1f}s ({len(all_paths)/elapsed:.0f} files/sec)')
print(f'  OK:      {status_counts["ok"]:>7,}')
for status_type in ['shape', 'dtype', 'value', 'corrupt']:
    count = status_counts.get(status_type, 0)
    if count > 0:
        print(f'  {status_type:7s}: {count:>7,}')

if failures:
    print(f'\nFirst 10 failures:')
    for status_type, msg in failures[:10]:
        print(f'  [{status_type}] {msg}')
    raise ValueError(f'{len(failures)} tensor integrity failures')

print(f'\nTensor integrity: PASS')

## 9. Class Distribution

In [ ]:
# ── 6a. Per-class frame counts ────────────────────────────────────────
for split in ['train', 'val']:
    print(f'\n=== {split.upper()} ===')
    total = 0
    for cls in sorted(file_inventory[split]):
        n = len(file_inventory[split][cls])
        total += n
        print(f'  {cls:30s} {n:>6,}')
    print(f'  {"TOTAL":30s} {total:>6,}')

# ── 6b. Check all expected classes are present in both splits ────────
for split in ['train', 'val']:
    present = {cls for cls, pairs in file_inventory[split].items() if pairs}
    missing = set(class_names) - present
    if missing:
        print(f'\nWARNING: {split} missing classes: {sorted(missing)}')
    else:
        print(f'\n{split}: all {len(class_names)} classes present')

# ── 6c. Split ratio ──────────────────────────────────────────────────
val_pct = 100.0 * val_count / total_count
print(f'\nSplit ratio: train={train_count:,} ({100-val_pct:.1f}%) / '
      f'val={val_count:,} ({val_pct:.1f}%)')

# ── 6d. Frames-per-scene histogram ───────────────────────────────────
scene_frame_counts = Counter()
for split in ['train', 'val']:
    for cls, pairs in file_inventory[split].items():
        for _, _, scene_id in pairs:
            scene_frame_counts[scene_id] += 1

counts = list(scene_frame_counts.values())
print(f'\nFrames per scene:')
print(f'  Min: {min(counts)}, Max: {max(counts)}, '
      f'Mean: {np.mean(counts):.1f}, Median: {np.median(counts):.0f}')

fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(counts, bins=30, edgecolor='black', linewidth=0.5)
ax.set_xlabel('Frames per scene')
ax.set_ylabel('Count')
ax.set_title('Frame count distribution across scenes')
plt.tight_layout()
plt.show()

## 10. Normalization Stats Recomputation & Verification

Recomputes RGB and depth normalization statistics from the **train split**
using vectorized batch Welford, then compares against `norm_stats.json`.
Uses multiprocessing for I/O-bound tensor loading.

In [ ]:
def compute_stats_chunk(rgb_paths):
    """Compute partial Welford stats for a chunk of files.

    Returns (rgb_n, rgb_mean, rgb_m2, depth_n, depth_mean, depth_m2)
    as numpy arrays for merging.
    """
    rgb_n = np.zeros(3, dtype=np.int64)
    rgb_mean = np.zeros(3, dtype=np.float64)
    rgb_m2 = np.zeros(3, dtype=np.float64)
    depth_n = np.int64(0)
    depth_mean = np.float64(0.0)
    depth_m2 = np.float64(0.0)

    for rp in rgb_paths:
        dp = rp.replace('_rgb.pt', '_depth.pt')

        # RGB
        rgb = torch.load(rp, weights_only=True).numpy().astype(np.float64)
        for c in range(3):
            pixels = rgb[c].ravel()
            n_pix = len(pixels)
            b_mean = pixels.mean()
            b_m2 = pixels.var() * n_pix
            new_n = rgb_n[c] + n_pix
            delta = b_mean - rgb_mean[c]
            rgb_mean[c] += delta * n_pix / new_n
            rgb_m2[c] += b_m2 + delta ** 2 * rgb_n[c] * n_pix / new_n
            rgb_n[c] = new_n

        # Depth
        d_raw = torch.load(dp, weights_only=True).numpy().astype(np.float64).ravel()
        valid = d_raw[d_raw > 0] / 1000.0
        if len(valid) > 0:
            n_v = len(valid)
            b_mean = valid.mean()
            b_m2 = valid.var() * n_v
            new_n = depth_n + n_v
            delta = b_mean - depth_mean
            depth_mean += delta * n_v / new_n
            depth_m2 += b_m2 + delta ** 2 * depth_n * n_v / new_n
            depth_n = new_n

    return (rgb_n, rgb_mean, rgb_m2, depth_n, depth_mean, depth_m2)


def merge_welford(a, b):
    """Merge two partial Welford states."""
    a_rgb_n, a_rgb_mean, a_rgb_m2, a_dn, a_dm, a_dm2 = a
    b_rgb_n, b_rgb_mean, b_rgb_m2, b_dn, b_dm, b_dm2 = b

    # RGB channels
    rgb_n = a_rgb_n + b_rgb_n
    rgb_mean = np.zeros(3, dtype=np.float64)
    rgb_m2 = np.zeros(3, dtype=np.float64)
    for c in range(3):
        if rgb_n[c] == 0:
            continue
        delta = b_rgb_mean[c] - a_rgb_mean[c]
        rgb_mean[c] = a_rgb_mean[c] + delta * b_rgb_n[c] / rgb_n[c]
        rgb_m2[c] = (a_rgb_m2[c] + b_rgb_m2[c]
                      + delta ** 2 * a_rgb_n[c] * b_rgb_n[c] / rgb_n[c])

    # Depth
    dn = a_dn + b_dn
    if dn == 0:
        dm, dm2 = np.float64(0.0), np.float64(0.0)
    else:
        delta = b_dm - a_dm
        dm = a_dm + delta * b_dn / dn
        dm2 = a_dm2 + b_dm2 + delta ** 2 * a_dn * b_dn / dn

    return (rgb_n, rgb_mean, rgb_m2, dn, dm, dm2)


# Split train paths into chunks for parallel processing
train_rgb = sorted(all_rgb_paths['train'])
chunk_size = max(1, len(train_rgb) // MAX_WORKERS)
chunks = [train_rgb[i:i+chunk_size] for i in range(0, len(train_rgb), chunk_size)]

print(f'Recomputing norm stats from {len(train_rgb):,} train samples...')
print(f'  {len(chunks)} chunks x ~{chunk_size} files, {MAX_WORKERS} workers')

t0 = time.time()
with multiprocessing.Pool(MAX_WORKERS) as pool:
    partial_stats = pool.map(compute_stats_chunk, chunks)
elapsed = time.time() - t0

# Merge all partial results
merged = partial_stats[0]
for ps in partial_stats[1:]:
    merged = merge_welford(merged, ps)

rgb_n, rgb_mean, rgb_m2, depth_n, depth_mean, depth_m2 = merged

# Finalize
rgb_std = np.sqrt(rgb_m2 / (rgb_n - 1))
depth_std = float(np.sqrt(depth_m2 / (depth_n - 1)))

recomputed = {
    'rgb_mean': (rgb_mean / 255.0).tolist(),
    'rgb_std': (rgb_std / 255.0).tolist(),
    'depth_mean': [float(depth_mean)],
    'depth_std': [float(depth_std)],
}

print(f'\nCompleted in {elapsed:.1f}s ({len(train_rgb)/elapsed:.0f} files/sec)')
print(f'\n{"":25s} {"Stored":>12s} {"Recomputed":>12s} {"Diff":>10s} {"Status":>8s}')
print('-' * 70)

all_pass = True
for key in ['rgb_mean', 'rgb_std', 'depth_mean', 'depth_std']:
    for i, (stored, recomp) in enumerate(zip(norm_stats[key], recomputed[key])):
        diff = abs(stored - recomp)
        ok = diff < STATS_ATOL
        label = f'{key}[{i}]'
        status = 'PASS' if ok else 'FAIL'
        if not ok:
            all_pass = False
        print(f'  {label:23s} {stored:12.6f} {recomp:12.6f} {diff:10.6f} {status:>8s}')

if all_pass:
    print(f'\nNorm stats verification: PASS (all within {STATS_ATOL})')
else:
    print(f'\nNorm stats verification: FAIL')
    raise ValueError('Stored norm_stats.json does not match recomputed values')

## 11. Depth Statistics

In [ ]:
def depth_stats_chunk(rgb_paths):
    """Compute depth statistics for a chunk of files.

    Returns (total_pixels, zero_pixels, all_zero_frames,
             depth_min_mm, depth_max_mm, sample_depths_meters)
    """
    total_px = 0
    zero_px = 0
    all_zero_frames = 0
    d_min = 65535
    d_max = 0
    # Subsample depths for histogram (keep memory bounded)
    sampled = []
    rng = np.random.RandomState(42)

    for rp in rgb_paths:
        dp = rp.replace('_rgb.pt', '_depth.pt')
        d = torch.load(dp, weights_only=True).numpy().astype(np.int32).ravel()
        total_px += len(d)
        n_zero = int((d == 0).sum())
        zero_px += n_zero
        if n_zero == len(d):
            all_zero_frames += 1
            continue
        valid = d[d > 0]
        d_min = min(d_min, int(valid.min()))
        d_max = max(d_max, int(valid.max()))
        # Subsample for histogram
        if len(valid) > 0:
            idx = rng.choice(len(valid), min(50, len(valid)), replace=False)
            sampled.extend((valid[idx] / 1000.0).tolist())

    return (total_px, zero_px, all_zero_frames, d_min, d_max, sampled)


# Run on all files (both splits)
all_paths_flat = all_rgb_paths['train'] + all_rgb_paths['val']
chunk_size = max(1, len(all_paths_flat) // MAX_WORKERS)
chunks = [all_paths_flat[i:i+chunk_size]
          for i in range(0, len(all_paths_flat), chunk_size)]

print(f'Computing depth statistics from {len(all_paths_flat):,} frames...')

t0 = time.time()
with multiprocessing.Pool(MAX_WORKERS) as pool:
    chunk_results = pool.map(depth_stats_chunk, chunks)
elapsed = time.time() - t0

# Aggregate
total_px = sum(r[0] for r in chunk_results)
zero_px = sum(r[1] for r in chunk_results)
all_zero_frames = sum(r[2] for r in chunk_results)
d_min = min(r[3] for r in chunk_results)
d_max = max(r[4] for r in chunk_results)
all_samples = []
for r in chunk_results:
    all_samples.extend(r[5])
all_samples = np.array(all_samples)

zero_frac = zero_px / total_px * 100
print(f'\nCompleted in {elapsed:.1f}s')
print(f'  Depth range (mm):      [{d_min}, {d_max}]')
print(f'  Depth range (m):       [{d_min/1000:.3f}, {d_max/1000:.3f}]')
print(f'  Zero-pixel fraction:   {zero_frac:.2f}%')
print(f'  All-zero frames:       {all_zero_frames}')

# Sanity checks
assert d_max < 65535, f'Depth max {d_max} is at uint16 ceiling — likely corrupt'
assert d_max < 20000, f'Depth max {d_max}mm (>{d_max/1000:.1f}m) — unusually deep for indoor'
assert zero_frac < 50, f'Zero fraction {zero_frac:.1f}% — too high, possible preprocessing error'

if all_zero_frames > 0:
    print(f'  WARNING: {all_zero_frames} frames with all-zero depth (sensor dropout)')
else:
    print(f'  No all-zero depth frames')

# Depth histogram
if len(all_samples) > 0:
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.hist(all_samples, bins=80, edgecolor='black', linewidth=0.3,
            density=True, color='steelblue', alpha=0.8)
    ax.axvline(norm_stats['depth_mean'][0], color='red', linestyle='--',
               label=f'mean={norm_stats["depth_mean"][0]:.2f}m')
    ax.set_xlabel('Depth (meters)')
    ax.set_ylabel('Density')
    ax.set_title('ScanNet Depth Distribution')
    ax.set_xlim(0, min(8, d_max/1000 + 0.5))
    ax.legend()
    plt.tight_layout()
    plt.show()

print(f'\nDepth statistics: PASS')

## 12. Dataset Loader Smoke Test

Verifies the actual PyTorch Dataset/DataLoader pipeline works end-to-end.
Requires the `src/` package to be importable.

In [ ]:
# Add project root to path if needed (for Colab)
PROJECT_ROOT = os.path.dirname(os.path.abspath(DATA_ROOT))  # adjust if needed
# For typical Colab setup where the repo is cloned:
for candidate in [
    '/content/Multi-Stream-Neural-Networks',
    os.path.join(os.getcwd(), '..'),
]:
    if os.path.isdir(os.path.join(candidate, 'src')):
        if candidate not in sys.path:
            sys.path.insert(0, candidate)
        break

try:
    from src.data_utils import get_scannet_pretrain_dataloaders

    train_loader, val_loader, num_classes = get_scannet_pretrain_dataloaders(
        data_root=DATA_ROOT,
        batch_size=16,
        num_workers=0,    # 0 for smoke test (avoids Colab multiprocessing issues)
        crop_size=224,
        normalize=True,
        balanced_sampling=False,
    )

    assert num_classes == EXPECTED_NUM_CLASSES, (
        f'Loader reports {num_classes} classes, expected {EXPECTED_NUM_CLASSES}'
    )

    # Pull one batch
    rgb, depth, labels = next(iter(train_loader))
    print(f'\nTrain batch:')
    print(f'  RGB:    {rgb.shape} {rgb.dtype} [{rgb.min():.3f}, {rgb.max():.3f}]')
    print(f'  Depth:  {depth.shape} {depth.dtype} [{depth.min():.3f}, {depth.max():.3f}]')
    print(f'  Labels: {labels.shape} {labels.dtype} [{labels.min()}, {labels.max()}]')

    assert rgb.shape == (16, 3, 224, 224), f'Unexpected RGB shape: {rgb.shape}'
    assert depth.shape == (16, 1, 224, 224), f'Unexpected depth shape: {depth.shape}'
    assert rgb.dtype == torch.float32
    assert depth.dtype == torch.float32
    assert labels.min() >= 0 and labels.max() < num_classes

    # Check that normalization was applied (values should span negative to positive)
    assert rgb.min() < 0, 'RGB min >= 0 — normalization may not be applied'
    assert depth.min() < 0, 'Depth min >= 0 — normalization may not be applied'

    rgb_val, depth_val, labels_val = next(iter(val_loader))
    print(f'\nVal batch:')
    print(f'  RGB:    {rgb_val.shape} {rgb_val.dtype}')
    print(f'  Depth:  {depth_val.shape} {depth_val.dtype}')
    print(f'  Labels: {labels_val.shape} {labels_val.dtype}')

    print(f'\nDataloader smoke test: PASS')

except ImportError as e:
    print(f'Skipping loader smoke test (import error): {e}')
    print('This is expected if src/ is not on the Python path.')

## 13. Visual Spot Check

One random sample per class from the train split.

In [ ]:
import random

random.seed(42)
np.random.seed(42)

n_classes = len(class_names)
fig, axes = plt.subplots(n_classes, 2, figsize=(8, 2.5 * n_classes))

for i, cls in enumerate(class_names):
    pairs = file_inventory['train'].get(cls, [])
    if not pairs:
        axes[i, 0].set_title(f'{cls} (no samples)')
        axes[i, 0].axis('off')
        axes[i, 1].axis('off')
        continue

    rgb_path, depth_path, _ = random.choice(pairs)
    rgb = torch.load(rgb_path, weights_only=True).numpy().transpose(1, 2, 0)
    depth = torch.load(depth_path, weights_only=True).numpy().squeeze().astype(np.float32)

    axes[i, 0].imshow(rgb)
    axes[i, 0].set_title(f'{cls} — RGB')
    axes[i, 0].axis('off')

    # Mask zeros for better visualization
    depth_vis = np.ma.masked_equal(depth, 0)
    axes[i, 1].imshow(depth_vis, cmap='viridis')
    axes[i, 1].set_title(f'{cls} — Depth (mm)')
    axes[i, 1].axis('off')

plt.tight_layout()
plt.show()

## 14. Summary

In [ ]:
print('=' * 60)
print('SCANNET PRETRAIN DATASET VERIFICATION SUMMARY')
print('=' * 60)
print(f'  Data root:          {DATA_ROOT}')
print(f'  Classes:            {len(class_names)}')
print(f'  Train frames:       {train_count:,}')
print(f'  Val frames:         {val_count:,}')
print(f'  Total frames:       {total_count:,}')
print(f'  Scenes:             {len(scene_to_split):,}')
print(f'  Split ratio:        {100-val_pct:.1f}/{val_pct:.1f}')
print(f'  Tensor size:        {EXPECTED_SPATIAL}x{EXPECTED_SPATIAL}')
print(f'  Depth range (m):    [{d_min/1000:.3f}, {d_max/1000:.3f}]')
print(f'  Zero-depth frac:    {zero_frac:.2f}%')
print(f'  Class-folder match: Yes')
print(f'  Frame uniqueness:   Yes')
print(f'  Data leakage:       None')
print(f'  Norm stats match:   Yes (within {STATS_ATOL})')
print(f'  Tensor integrity:   {status_counts["ok"]:,}/{total_count:,} OK')
print('=' * 60)
print('ALL CHECKS PASSED')
